# Lección 12 - Reducción del Historial de Chat con el Bloc de Notas del Agente

Este cuaderno demuestra cómo gestionar el contexto en conversaciones largas usando Microsoft Agent Framework. A medida que las conversaciones crecen, el conteo de tokens aumenta, eventualmente superando la ventana de contexto del modelo. Abordamos esto con un **patrón de resumen de contexto** y un **bloc de notas del agente** para memoria persistente.

## Lo que aprenderás:
1. **Por qué importa la gestión del contexto**: Comprender los límites de tokens y las ventanas de contexto
2. **Agentes conscientes del contexto**: Construir agentes que gestionen su propio contexto de conversación
3. **Patrón de resumen de contexto**: Usar herramientas para condensar el historial de la conversación
4. **Bloc de notas del agente**: Memoria persistente que sobrevive a la reducción del contexto

## Requisitos previos:
- Configuración de Azure OpenAI con variables de entorno configuradas
- Comprensión de conceptos básicos de agentes de lecciones anteriores


## Configuración


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Por qué importa la gestión del contexto

Cada LLM tiene una **ventana de contexto** finita: el número máximo de tokens que puede procesar en una sola solicitud. A medida que avanza una conversación de múltiples turnos:

- El **conteo de tokens crece linealmente** con cada mensaje del usuario y respuesta del asistente.
- Los **tokens del prompt dominan el costo** porque todo el historial se reenvía en cada turno.
- Eventualmente, la conversación **excede la ventana de contexto** y el modelo o bien trunca o genera un error.

### Estrategias para gestionar el contexto

| Estrategia | Cómo funciona | Compromiso |
|---|---|---|
| **Truncamiento** | Eliminar los mensajes más antiguos | Se pierde el contexto inicial |
| **Resumen** | Condensar los mensajes antiguos en un resumen | Se pierde algo de detalle, pero se retienen puntos clave |
| **Cuaderno / Memoria externa** | Almacenar hechos clave fuera de la conversación | Requiere llamadas a herramientas, pero sobrevive a cualquier reducción |

En este cuaderno combinamos el **resumen** con una **herramienta de cuaderno** para que el agente pueda mantener la continuidad incluso cuando se condensa el historial de conversación.


## Creación de un Agente Contextualmente Consciente


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Simulando una conversación larga

Vamos a recorrer una conversación de múltiples turnos para ver cómo se acumula el contexto. El agente debe retener detalles clave (preferencias, presupuesto, fechas de viaje) a lo largo de los turnos y demostrar continuidad.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Observe cómo el agente retiene el contexto de turnos anteriores: sabe sobre Japón, sushi, templos, fotografía, el presupuesto de $3000, viaje en solitario y el período de abril. En una conversación corta esto funciona bien, pero a medida que la conversación crece, reenviar todo el historial se vuelve costoso.

Continuemos la conversación con más turnos para ver la acumulación de contexto:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Patrón de Resumen de Contexto

A medida que la conversación crece, podemos usar una **herramienta de resumen** para condensar el contexto acumulado en un formato compacto. El agente llama a esta herramienta para registrar las preferencias clave de modo que, incluso si se eliminan mensajes antiguos, se preserve la información esencial.

Este patrón es la base para una reducción de historial más sofisticada:
1. El agente identifica hechos clave de la conversación
2. Llama a la herramienta de resumen para almacenarlos
3. Los mensajes más antiguos se pueden eliminar de forma segura porque el resumen captura lo que importa

A continuación definimos una herramienta `summarize_preferences` que el agente puede llamar para registrar un resumen compacto de lo que ha aprendido.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Resumen

En esta lección aprendiste cómo manejar el contexto en conversaciones de agentes de larga duración usando Microsoft Agent Framework:

### Conceptos clave
- **Las ventanas de contexto son finitas** — cada token en el historial de la conversación cuesta dinero y cuenta para el límite.
- **Las herramientas de resumen** permiten al agente condensar el contexto acumulado en resúmenes compactos, reduciendo el uso de tokens mientras se preserva la información esencial.
- **Los blocs de notas del agente** proporcionan memoria externa persistente que sobrevive a cualquier reducción de la conversación.

### Lo que construiste
- Un **agente consciente del contexto** que mantiene la continuidad a lo largo de conversaciones de múltiples turnos
- Una **herramienta de resumen** (`summarize_preferences`) que registra detalles clave del usuario en un formato compacto
- Una **conversación de múltiples turnos** que demuestra la retención del contexto y el manejo de cambios

### Aplicaciones en el mundo real
- **Bots de servicio al cliente**: recuerdan preferencias a lo largo de sesiones largas de soporte
- **Asistentes personales**: rastrean proyectos en curso sin necesidad de reexplicar el contexto
- **Tutores educativos**: mantienen el progreso del estudiante a través de múltiples interacciones

### Próximos pasos
- Implementar una herramienta completa de bloc de notas con persistencia basada en archivos
- Agregar truncamiento automático del historial después del resumen
- Combinar con bases de datos vectoriales para búsqueda semántica de memoria
- Construir agentes que puedan reanudar conversaciones días después con el contexto completo


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por lograr precisión, tenga en cuenta que las traducciones automáticas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional realizada por humanos. No nos responsabilizamos por cualquier malentendido o interpretación errónea derivada del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
